# Independent verification of g₃(7) = 474 (Erdős Problem #817), self-contained

This version needs **no GitHub access and no token**. It contains compressed copies of the verification program,
the search program and the published count table, taken from commit `8829c7f9c56d` of the repository. Cell 1 unpacks them
and prints their SHA-256, which you can compare with the files in the repository. A CPU runtime is enough.

| step | what it checks | time on a standard Colab CPU runtime |
| --- | --- | --- |
| 2. quick | certificates (from the definition), Lemma 1, g₃(5) = 60, g₃(6) = 168, exhaustive search at N = 473 and 474 | ~10–15 min |
| 3. critical | exhaustive search for **every** N = 419…478, compared with the published count table | a few hours |
| 4. full (optional) | every N = 1…478 (does not rely on Korsky's bound g₃(7) ≥ 419) | +1–2 h |
| 5. Rust (optional) | the independent Rust implementation at N = 473, 474 | ~15 min |

Each value of N is logged separately. If the runtime disconnects, run cell 1 again, then the same step; finished
values are skipped, as long as the logs are on Google Drive (step 3).

## 1. Unpack the programs and data

In [ ]:
import base64, gzip, hashlib, os
ROOT = '/content/g3verify'
FILES = {
  'Makefile': ('330511c38aff5fe808422b43b1658bdd9ca0a3934942d36e02b5f51c76ffdc60', (
    'H4sIAAAAAAACA5VVXW/bOBB8Dn/FIA1SG7VoJE6Qq3q+u9TXL6CJg6QPd08xTdMWYYnUkVRSA/3xXeojddoU574IXnJ3ZnekHT/D'
    '60rnC4g8R+nsyonCc+Ba/Vdppwplgk8hMIG0Ralz5dArxFo991iopajygJmUswE0VxwrKWEdZC7M6hXsnXJOLxTudcjYM8xiISaT'
    'Med81h/guvIBf4xxxM9O6xxI4VYWS4IImYI2C1UqepgAgtJLrdwAV5uQWYPRAMIscFkVVxv03Fa7PHwOfaKLMA2lzJRcz2iq1Wgp'
    'fDiuK1cjGndJE6HyquZ7ffHhGFdv/vlEzD64SgZNRPeZMkgK4WQ2NiLoOwVlxDxXHjrUSMIQW2zgTuSx19K6EBNIMhq+FDSYJXx3'
    'r4mop/iK4/z6AkOclyVl3ehcS2tSNPK8/Xj+7ma8n0xHxCrLqiXd73PW3OHPMZrbrZ4YoxeYYq7NsBty6/etsc7aoj3yKhY2wfr2'
    'cVTrvGkTO4GaqLlijKKU7RXrhXZIynhZn3VcKbyTXcAlvtQZewe9yaQPetYz9JFYHPyFg98f1bZ97grx9+X0ejq9+AGrGalDaaKd'
    'O+kUacu78BfqG6HS5pvdfDvZXYxW+G6CNty9/vsO2oNb54cRsRDacOefup7EDeTBFnnHJRdP5OHwsF3Wee0eSeJUroRXlF8+hRso'
    'WYVhm/VwRW2zuKr0omlbjNc+KCM3zcZ69MTcVgHHKLSpgvL9NLqONrrezDsyqs6yBuRQMY6eMoBULpBdSEE1A6yNvadskVfKsxp5'
    'x0XZXoanF4XtzYXPQDTBD11lbpvGuc/iWFsGlnQGRuYR3IY8QpuQPmoUL2oXWiipfXSZpmFcYoyTM7K7k7MT9E6To9MHNV4RRzSv'
    'WVl74qgTvpNdefJnXpKcjhSTIp+hl9nK+T6ChVMJdQxF2ZuG5egl5ydnv7H282F7/4dLnifXjEl6qdETXIHELaNCP/8EGONX76eX'
    '/6b1P04tV5uMGoZ9BbKhfoaRBgAA'
  )),
  'src/g3fast2.c': ('0b524abdf5e63e82e94581ea27090b9966172662777d8c1941d1e2e4ba51fb9f', (
    'H4sIAAAAAAACA7VabXPbyA3+7l+BXiY3pPUSiXac1LLcSeJk5qaKnMlLe1PV1dDSyuKZIjUkZctJ3N/eB9gluZQox3ed010saokF'
    'sMCzABbks/092qerg5mfZl57Qq0WxcssWARf1ZScD29//UwTP5oGUz9TlOIydUmt5/4qzYIbjCg/mcxpFid0NT5wIpect8k0TulD'
    'El+GakFPXnZfNOn87S+f6NXBX//6wuu4bUhkoZ/8haKFn80V/gSTlCCHUh40XLNEKfJTaKcHoJ6TYggzaBIvFirKUv6RKPeYGRK9'
    'In+6CNI0gGyik/4pURTj/+irSmKaUBDRt5bXbnv3/4noNsjmlK4WNBkH5ONfnzqazdl4gB/f+Ob4W3DSH9wXNMdVLgO6bxKEToPo'
    'itYUpBSqKz+kYDbDzyjOQMvceGmeNcA2oM9zXv86WECHIc9dhv4EZp8FSZr1ZJkxL48ULCmL9ROsfB6nKmIlgmiSKD9l0XEyVUlu'
    '2DNIh1GiiUpplsSLigGXKoG3Fj7uUhyFd1rOTZAGGURrk0NZbeNVLjSYQn4w8cPc0i1ZFnRmHuxFlQR+GKTgEUcy+zaIpvEtxTNa'
    'xmAexBE7y88YTwT0hCFdKsICphTydMOYP6G6USFFLY8ipaYpjVrDJh0OL3rmzqAcb3kO6FoDVygaxa8LIieIwDZVE16YOBtqlUKy'
    'ZAWkrZbLOMnAydtnbw+aZC4uBKey0ssgSxUMMfGT5I4ESkvj8jSWtabzYJZRGMdLba7LBPadt2ZsTcdfZXHrBmrECdunwP8wnuY2'
    'PgYypiodXV8Ad9FqcQmnw3AWmK9bosK34T2t6DU5319/B+l1q+vqpQFHGD7p07AFE1w3ui6LcFi5s3eftH9T7Fx/ksFfGE6V2yZ6'
    'h417zTIxBGcGsp/qNYi0BoW0Yb6OVepfqeM8hoDVMIxpOA9olGbxcuxnYwF0v3PBI2rZ7+Iins3ATsYWQcQAx3DpHFomMeCbAsFD'
    'qAeODT2jaV03mFuT2u22LHweWPMNUzqlLjCWZkkw0cFic0tlcaEBeLegmwNMqKR1Cc9MTShSGuds02d7T7DvwhV8d5Jm0yBuz0+r'
    'Q2FwuTmWACtbdIBndQxhV/HI3pOpmkElev/q1yF1O3vZ3VJhiFaYcnQ4zmh1dNjb23u2T9gO0Zjx6azdY1negSd4ZeetCbuNKcod'
    '2ARyJ9fYD+CkoRtij879cAYwfGFzv37/i0cS92/nmOnf+EHow/+9Pd4zmMD7hQfoBpsLPBGTOWbDyKsQ7vFppm5pEXPQiLDy1UQE'
    'u4ytVWp2YrzKtCBHta/a9Orje7eJGCb3WE7rbHg+FiUcTi1vPnxJWR3wlEGwWgQTIASbZqrTBpbRxG5G1kDCePX+jP4F5bvPPIBc'
    'XDYjbdOpMx6z5PHYpZ9/pr/ko0aea7kjWAAX7Dr2SZpxjsKSQnYM7G+Znn+tXeSLRGWrJKLxUq2zMUaddZM66+cbny+Dgduj+70n'
    'KkzVoxizSeDNn/v13Hrmfp+cNX3nP6eAvYsFgv5g47OL3jP0nXfV/3bRH+b0nXfv7H+76F8W9EzFtPp7p/5H1oScuDLBWHvd22Nb'
    'okaZ7ZXWzCjCdu7ZAx97VPfBLsq3By0ZWohaCcMq442EmVO1xo0GfSSnkQd+l1Fl8f7w6uyfvXreJlPcIkVzFCHlA6QpEuoGi8/n'
    'n3exmN+lnHsNDwQnkwdLBoyU/bPL1WwkMaNB3YvK2m/DuLzTxF4KLEIthZkT8taVonkcis4o+vwe4zG5Q3rAACOWTSQ5sKo+Z+Sq'
    'EB7ZkGPbWkSlQgVr1y2K2XJcrllTGEMb+aMTp6HwainSOKwYQ9KSLrLKWPvPX85+dZaoX/HBF9ztMgyP+JK96xaUr0vKnPBnOjpw'
    '9zb2Mou6UhlA5EwQATPjpKbcWObb2qCYixXXcc5GRo8LFm4kMf/uymWcc8yfptkojNvtecDVgpMmE2wa/D058fXF6amfj3jFkOe7'
    '+awejxRuRCwWNHLSYIcbDJRuuImDKSqHJeKsRKT9PJ+yJk2y1lbcAH+9TF9/hbH+ngf5ssUPXegPitTDt4fc7feKe7d8DwTsgiZd'
    'ml9sZ6CX6THJ3DO/+J6ZPyMHMxDeL71cHn84lbCdiUv9MO7hG3XDPMBFo2ET8ocXdMOck8kIJqvcu6Hv2vCjAOXhLWqGkxPo6HLw'
    'KkfxpytudMCqxffd3WwawgbEFTY8KsBnAY9hA7me1sarauNtauP9QBtPa+NVtfE2tdliw+gUYN6U4/dyda/Dx5/lD3j8b9iTf8Ar'
    'Lk511rQf8f5drip5N3bx9qp6P95/Fb29H/L+XU6t6L3J+wEfmxiVl6Dl0Q+lv58gXMhXI1SRy/fPqHXKEWjUabeRMjDMEDg63Pc6'
    'lUQgAVUCkT5W1kRUw1/CjULyl3tgbUec2w6Ulggr1Cg6LzHwuhwoA1B0y8GVFWpwcJdgY4WXnXGlIzDGdINiXpwO1GcjiG9QoH0g'
    'LsiHNh2Qb6qHt8xOUYUkw6VYVIITUV+sXI2WGNfTsWrBGUpNpDeY0AXGoZcQ4E7P9u+Z9inn3QzO41qJRh3xopv79ULbMHeq+6BX'
    'x1z6/gHX8u9ssRw99y7qazypPCw9DmGF7vODI9YyAv4xfPgS5V2XrlY+yiCuhUTVEg2cZRxOUnWQMKA8a+aqGsImq+WWdmY2yOWu'
    'aBvxdmQvPgJzD/q9hAcbAofvvnWGYEmsTXBhBWqmmwf1dIJFi7ZAFRgDspgGPBx47p+GrbLfuPDTa1l6hANVfmSnr4KzrXByLOU6'
    'wKIyab997et7qDpwJPza9L7m7bdtDLL+hdj0sQjkGTYE1cg7vHiMN7fgIjyFnU2gt8OZRpNNqaw4NcG/AkSPhAlL0j79r5Nff4f6'
    'FYyw78CmT9pvXE45uXeRHvJ5m960acSphVCo1ejTeHy5CkL4YbyMl9L9CsNciQqmTGGMaQIMuxhNFDchZKF32inFKZmL/UZD81ni'
    '7J7NnJ8+nQ++fP7lfEjD/tMpffsJB0O3xl5dU4BE0PvA2Cxn8XTaxDQ5jBRqWvfk//t/R6CBQmtLwCxcpXMklymHK3shLHU6S0X6'
    'wCDL53q6PLikq0W+LBuRHOD5pDe4KDfZQBwFxaXgfRLMuF00PP94fv6+dCibi73Tp6F4tCUSe6YRsYPMs8jkjG3jQ+hOOKAZb3Wq'
    'MeZuUm6JgrdGrLXdgHCWwIGnKaLxd+JuzNK76a5uO1U8eWegf8fY51nArYg8FQ/fiVtx/g0VuE1Gd3DnRqUpXFjFEqqT7CtQasir'
    'tZAe5G1griqgL/hxaDRrZP3hR9awAUFVYn2cHUgQznFsM6r6725nS2MtQfKu0TVNTZPLbOf4E/GO5Kiqm34cwfJPGcnutPvEe+Ds'
    '9rZU+o0ds6lI4Te+2/J6+MYW5O/tI0COut8Y7CglsCngnJXqbVFtqAVL/cYGb27m5FpN6mIn24qDZF+MtXn+uN97NKPtFbErZgnH'
    'Yi2kV7vmByIxJj86CBcG0htgltRplLu/bg9gRq92gtYCf2vl5Sy5yXdnbwLeA9kOeizb9E9MDkSU//6dNsZ4G9SON/T43o6KsIYP'
    'f+/ipe89BLrq9vW2tm/+MdmLE4W7e+Vlk6oIrjV2vd+Bw/KqGpgr5ZofRNzU04nDNFUlxeB8VyQZ03ezO+hvvnw+f/dO7FS2zznV'
    'uHtmq2uKY9061I/3ivJtsQJ3P0xjCgOuvPSjxlWSKLsIkzLOOdOPF6+S+DbFYU0/bcuXVikK1+aRoM8hkgNtsb5W18o1LI+78hwT'
    '2zkjrfyxfkhmdLYefLEm7HVjriFxuRJH0pE1j0j/Mb52hlCQt/4kidO0NZmryXWqnz4U4H84+W6QlA6qScCMkPWO5CtPDSbVWnT9'
    'cNJdS1xcT6y6cq2D//rB08jahLg1x7g1B7m1nV+hxbqaX2VSXVwxpCXC9QDHFHNViSt5LNnMqWuOJ+usl5dAGogW0yhrtXadETfy'
    'lYUa3dYEEv3L+EZBspXFxBVs3pOKyypuqVZYDhex7loq242ialf6l8pzwBX7uhy0KkVujK0WvHbskqUfmYZqWke8lBNii8matJRj'
    'IF9XK38mOsmb+lof18ysDG7MArfTvPFfzBIRlcGqI4WrdGWkCw4p2EoyyRqcBxY44J1rpZYSOoLoxk8CP8qOq/3sURg3uUVuuaqS'
    'mW/LNei6/7ZQjzMsI1p6koznouHV6dUz42Mxw/DUYkt/IzN6bI26W9IekGEx0x1SS0fplJYA0f156cifyRGEe+4Vo7GD+MQxMHse'
    'SHE3c0vl6NWR8wovkWEta/WTqwlOqnM/of19/LgpumtgzjdhtUN+FDozpyOcfFSSNOkn85bA0/T/eUFATlgsdtRhQxo9vZ7Rm2Hv'
    'Z3HgCEnRxWDFhwKy8qZ3AYwNBWTl4EE+o0y+fJ+XdUqH8GdJenjBHVKrd8OKl8TPK8TPhRg214sqyY4qZEc2T0CcX8SKIz88Lt5g'
    'QN4My1cX9JsM2RzbfcjvMVwqPuTkbz44lwBEK5+p8lcZmvopPOfVPQvJ9jsPctjMH9zr94OsV76SVSRP8mUcMdZfhVkhtM9xz0p5'
    '5bhZ8YvKil9ow5Tn2Jxe0prF1GqhMcK49IjAjR/71aJN3o8aHeJUrjEjhNuAYYZDiVncDazlxLelZIF1T0zX0DHvNqFAmqkkdSGi'
    'hvlHE4QjftdHRwLvpV4IP1DM+4lDfij2jNNXgw71bX4UnN/+yPMKAs9UovK8uZqLQ52KQ9aSl4tLDitSxYXSgJbazvXD4Aol2xhQ'
    'iifO0SHCQPBVxTM5PvA7Q46Ib9ALlvoSIy9du11i3scBgUZzD+DTr9/wVaMvO6HSbYKc6zGOE9x8lx+OFZUWagEmjiS+JnUKbWTA'
    'frjEnZ0dEXjH0r/lzI0NbPZmsbxUuI7jrNhIP8fnKCs/xRIt7S6uP6y6Gvszf3S98UTbSjgmVUpjhvm1oJpJhMUYly3DmkUNqGhF'
    'DXhfY4GDVotXlSfgi81kbI4wpZCCpEwbjZzkvpIYDBHC7Skgh+3VMpz1SO3W+IigE1O6AJLqdwB/xPadC/1cu+PKGdo+qr7W4xul'
    'TocVt6yi6yLJelYVWD26blU43mDAS82LHPNzo82xWets1TmbD+226pytGqdb00r5/TWOldXFhvxKhTFmkxe1ld/zgqFSLGwVClul'
    'i4Z8pdwx8rbKka25Oqrp2dZe+QGbEhxcjiAt5oG+xTlyaC1pGq/4BKZ4gzv6h+uYEALqrMMh6s3g/M3fP40/vP04/vT2TTk5789K'
    '1zfiP4ggK0lq/adhONWgOv5JrIlSmAOMW7MRp7o3PJXeML7tvjAxI3DQ+JzaB5iChF/r6z9tezPJRKoiY6M9XN8A4HJUlKPLRPnX'
    'uyq1/wFIg5Okyy0AAA=='
  )),
  'verify/check_set.py': ('3aaf2e807207856b1959cc213bf5073867ed54bedcfb5d07cc03293a3bf5e5cc', (
    'H4sIAAAAAAACA21UbW/iRhD+7l8x51OK3TMmQK65cscH615EpDRFIu2XKGct9hBWh9fu7pqEJvnvnVnbBNJDApvZ2Zlnnnlm3r4Z'
    '1EYPllINUG2h2tl1qcae7/tetsbsR2rQxtUO+n3AB5FZyFBbuZKZsAjOA1alBgGZULnM2Uo3IIHgq85LA3NdLjdYwNsPw/Mw9rzP'
    'fMVEcC/tuo0olcU7pBiabAVamQHFAlWCWZfaZrU1Ew8gECHYNcL4uwJTFwb4N5WA9BX0hQApFDyeRsNo9PxdhUARESoh9b00CLk0'
    'VqrMfuRYyyaWqZeEtk+BHOpZkIQwhUcX+VFwuMUz1TaBResK5Ypqe4asVFZIZRilKhX9NVYoC+O+RV1AMqck/Alq+ARb+t43Fdfw'
    'jl6nMNqGUUMf5rDcgUUGdwe4Rb1zmOkqsRRywjY1Fx17TAPTwyVwffhPLbdig5Sc4lxiUQgY8q28zMzgj6/Xsz+/LOIi/wjLkgDw'
    'lS6vVDlWSD/KbnbUm9qIO5x0GgCCIle7wZEMhtF4NIzOxu+js/Pz6OzDb9HZ76Po/ekZwE1Ratd8A3Ec33r4IAmzFbY2cApytWqL'
    '41IqYQyaBpGLT4Wx5GRRUcdBEom2LDemM5id8TwvxxURpZXQu7TrJnVs4rh2kpjCDT1JCL9S11iXGIFr47+yCrAi3SVh2BxUhu37'
    'THGly7ymeMFpRFXCiPqjiR1hp8QtZQlvXRqNttYK2MYZSS7T5g/axkD39oct5qZ/KZvSsajSlUbc454RaL4cun8MTTMwLdQdBk1q'
    '0syw9e58Fsfgs7KgCRZWkg6DhJAfuLskschzxhQswibRjMkyxC3mwawzUWcaMDPzAkdGpFrKhqouUNOA8+kxmns+n5kbyUgnt8e5'
    '5Yqn4B1r+QRGzNdpI+DWOBiQ1d1He3zzgO9vYmOwYZayR244DgPwqBz251rXh+5XpcK2GQWNbdDiL3+kYrOhmtl9X7DQd4yHNBfT'
    '6/bmqKKENUYbK7CNjnh98Y2YxLIRGQa9x14EvV74YnhuDabaSBv0ol7InNjYWE2qbHXVMtVJiWX6plEW9Z/yFNK9fSL2jkmqNKPx'
    'L67+Ti4vvsDF1fyva591fuS1r9QReXTEq0yq+sVo0/JHRL+ZYjn8ZOD2nmvnuWZPt9EVGiern+t9f61dClPw58li4TuFcFInC44Z'
    'AhJK8L8lF5e+96rSE1q605OcGvnAj2RKhqcOZrMGOqwToLOAnJpRfWoW/LifzPuMqTl+mj1RnPDE+K/EdwJBgzRq+xBxTvdsB4df'
    'D9iKjgj5n5TBd6V2PEnjZNmWujcnc0Llc/LWEoXhC3VtH3+Zvmas2YEkWV67Ae/bztfFH/IuIluaKlFgmvIY+mnKw5CmfqOoZjK8'
    '/wAL8RJwEggAAA=='
  )),
  'verify/bruteforce.py': ('70f0dd969a111e0722414f64b6452a0badeb5015f949d04d5eb40e770b2f5c4c', (
    'H4sIAAAAAAACA61WbW/bNhD+7l9xdRFMWm0ndppt8OoC7taiBRpnaLLsQxCotMTYTCTKIymnbpb/vucoyZKTtuuHCYksHY/38jx3'
    'Rz19sl9Ysz9Xel/qNa02bpnrw0632+3MTeHkVW5iOVhtqN+nRF4prZzKdT+Va5mS1yCvwneymUhTioWVlgKlE7mSuGlH+RW5JXSE'
    'dWSlMPFS2nDQ6fyVmxtLiTIydumGrkye0WuT5PYHyo1aKC3SllcK/Br9YfJ5KjN6+svw53DcIVxvg2lIE7ojW2TRnSCl6fSeBI3p'
    'FKK5lT6GKd1DNxAWK6evz1iktJMLaWzozUxJWVrkeUL0YvKyMhvn2gmlLekcfxqv1gkkddh30mQkjHLLTDoV08rkCyOt5VgLekFr'
    '/N/28PiMbhHdaD3ovHOU5IBndnJGhZUelruD3rA3uu8rfQ0c1Fq5DRlGPitSwYn/ijCtkyIh5QjYxQDNLYXb1QKyK2GEk+mmEzhp'
    'XST/LtRapFLHMiSbe2fuNocJ6MWIXlm/E3gYSdBUCbYnJBacryMp4iXl2GXA1RsQ7AUzjkLqIpPszNL0/XvS/RJly5heDAeD2WWN'
    'm9ILbBE6QbSr3EBlmd9SJvTGe2W0e7RKC05JdmIBiFUM3uO80FA+j26CWUgB/M2lYfOen5v/cBj6grwB6ljTPbpdKoSeFchKgCPZ'
    'uQVrZE28vzjkshwNYuzPVipF/n6t//vs5MPJybEP3UvWAOxqgw3lQ2QsYCmsWMhx3Ti1zm7vaJpl4hM1V7AyipPLtaRU4bZCarMy'
    'WFYtqxHXN832+y2CvVmDWNFCz0h+WgrkqtaS2jpcFaFvbpUxF2BSGpfnqa0FpYX6zW5sp9NBB1ZNFKG9LHqibDp+4aY7uPevDLjv'
    'vOm4Dr9U+Qc6FkGJckiwCsvLXUa6wpSCytVS2OhQrAIEXWBMlMYQ85kpJKmrapTwTJDEjf2xVPzYNOq290reWv3HqbM5bOHQLZKU'
    'Se2pXPFy6VhYiTJIgF9LwnkozgNwLWSQhU3GBZRZ8UJdbmWsft2oK8Qz7FF7F1+39c7ryx25uqLApxDSHo1oMqEDX5O1cH8fUhhf'
    '2117LXQZujbab+BGVnBzO0XzTdTM2S3BlbbO3ZaU3ToIKxuoIi3MJqonmGxqREpdAdpAJzncbekNMDaTInZBcNBjXEZhj0eFFG7C'
    'qMNLq57YVpEFUtGPJFRpTfX4ESY/q1Uge8RxtcArCw5x7MKzA8XWPtQGIkmCiug2fnWqu4M1cEbBxOTw4ADhY38yGY4Onx/VEGIY'
    'TaquGnzwPwErlebnGOmg0z8/bTXtmPggfTDhhs8v+dGqz5LnxPMtnDdNaQG+oxZcvDrdBRtDDod9OfaD7abhEUC/eVCQ6ivFQU8m'
    '32C8fXF+zzDVHi34+Rd0j9+dHk/PfnvbZdIqFKoZlgqDc5kLx24TjZpES9RbLnnUA+wBb2fbox791JTBcbUaL3MFzi5G4OqIy41J'
    'A3WXjeq0UrUiW6WygeiYu9aj1Pl/APoiOF8Fplp4OM0pU/jqcvxBNYYybO4Uri8wrrCyeMszNQKW0SzAqThrpmt5zva2R+b46+es'
    '//Sohi1OrWC+8SO5gSDcztlzWLsox5nN/cit3h7Xrfb4NhDF286o9V99TyWXOfR3TNW0voKLYNYLv6vKH1dz/OVahoEbBlk/3lHn'
    'PRAr/hQOpo3r81oW7xB23vMbKr4yIBxUkcANj0McyQO0xjqklzjR/EFQiy6GlxxHd+e7oNtElTO3j+ZXExLbkZ+UCw7YGbQlRiOA'
    '9Ao8xbkCW87KBf9p82BtdNkM+9kOy157l+gqZdh4XJ47/JeDHKrjL7XM6cn7P8/encxoNtlL6G7P3ndxXoJu6va6g+scSGZ8gDkD'
    'f2HYOiMqA36f5tv5mPYseyp8cUFUm0JQXXpo7TxEzXluEFvbMONhXZIXbnCFD9xlwOcloI0iLTIZRZ6tKGKWo6giqqS88y/X+NUb'
    'mA0AAA=='
  )),
  'verify/verify_result.py': ('c704749873003f7994cc4be0ed237635ace80e574725751c03fb34133b4193d0', (
    'H4sIAAAAAAACA61abXPbuBH+rl+BJqMhmaNoyfJLoqvaSRNfL21OzsRpplNH5VAkJLHmi0qQtjWZ9Lf32QVIkZLiuw+XmcikACz2'
    '9cHuQs//cFKp4mQRZycyuxebbbnOs3Hv2bNnvXtZxMutX0hVJaW32YrBQOSZHIR5mgZZJHg8DoMyzjORL8XKH9uXjpiKs8szscwL'
    'EYh1hZlmoiyEHWRbkQbhOs6keIjLtViF4UmYBNmqRxQ/8O5i/KMIEpWLosqUqLJwjXEZiTgTf83zVSLFmzwJFq5QUmra25OQvvH1'
    'ixdvttnC8Xq9SgUrOemJWqx69oFk/63i8E50/tn/Ox+MzkUaZ85vIBAWcQlVJOJ2MPhPvlDib3N6zKtSvH33cU701nlVqImYkYJG'
    'rzzv7PLlb6G8rJKEGfpNlEcN3SCKYLCiUqUo80afolxLKDKSG4mPrBQfaUJjoKDU/F2OXbainUkZKREGxSp3ej1W04SYWVRxggGi'
    'tinyVRGkyhXhWoZ3SixyGDaURam9QyphXw6ULLXF0+CRSLviZfe70fjipSOiuJBhmUDsIk+JfG9nkUgu4ywmb3NFATfUu72X8EYx'
    'ElmVQgqYINnSMLiKqhB7k1eek1deDAX5GL1f0Pvo4qWwmdeMRoV8LAuZBklrRzCoHJeXsS+SuPJxHUBn8b3EcFCE65bSoC+oOkpj'
    'peIF/JTFdnj9zMSFLR8DFpBoIZpoD8dt7Yno2gRFnK14Bls/zKuMjBSWeaG0xljx1SKJ1RqRUQa0m/YZdZJd+rxCeaG693q1a06+'
    'wz5FqoQHbDuuKWyaXVDkiSwvwQOmYKvFVvwdXNxtLVZHDpW5YpNUiqTjlS1ZHtZYI1SeVGQ1hc1jBWmNjKCWEBCAZNIAwjHBSH81'
    'EUxVkKxHimFPbPE+MpxHOexOTBcSisaahuMF9BIBGT6SNcEB6yylTSaCnIs3vQ+SCgSAaDNhk4mSfIVRMLJBiMwIhzj+HKag9yGJ'
    'qlJGXu/qMS6FKoMSKhmKeLk0DLK7ik2gFM0ifI3TTV6UILKCMpSs38M8C6uiQHB6y6qswKEIEILLZlzd14+rJF/Uz7mqn4qGlKoW'
    'iANEQTOmts1jGaey1/t4ff0JqsuVtwnKtYf4y4JU2t97DxaK/tq+Twrxfcdxep9e/+X9VYvGf/I4s4muKyzjkxYeO25pOb3372ZX'
    '/husK6RH6gM9u7D+PZvaX6IfHJGZv4331N/nkQTY2bfDwSsxxxckx5RfPbzWhD/+KuHPbSJ7u9RUbg6o3Fy//8end9czUZP78pWp'
    'uKDy5RvWGYmx8Hbe6/UAWoRGULhNinRFfucCycogTqaW5Uw4WswaL9gQLNvNTMfh4Q3goLSt/uBMib7qK0v0hW19eH1zY8HBME/I'
    'RElh/fT63XtoWq+2LUEHtiV+MNs5NFc/mvkWQnGJ0F1PPxWVdAy3am2HaeSKFy/uHhr+4IdZy508wKGeFT5EU23rMNiQt/qIDIQC'
    'k3RFCVQ1j0zP7BFuKn+ZBCtlmx0QD/Y9ZM8LsBSkMYH4OlD+Io1PHX0YnNDeJ1gZZ8v8R2HPEJqu2H2yfFUW3ENAhg1beiuP0o3r'
    'G8ejgKONymI7aRCKKJHjQuu21aFvOV4hg8jWBpCPodyU4vrmqijyYrfe6GWfEx7X0mj/0UgL99Ff+nH0Rb2Y4L/95QbO5jIjeist'
    '/N4yMGUGmnXR/jrS5t4y/o5W2N6L7mzDuFG5tyryamOPWIOGb/YQLRI5n969mcgzDau7mb1O/mSR7SxCS+ajWeupTRKXtqHBbDck'
    'av/g9MIuAY10Ak9aCt25B5GHwC1XMrYqiwDfWxa/7iSa4rvXFc6XDMfh61/eWnyu1PqeitPxzrANETF4O7v2P1z985PVFu65AAHx'
    'L5mJ0cnpRNC4iBWSxRA+BICKhK2S/MH5UVSQjM80IAB7JQ0LwkltNjAWaiVAUbfW4HosBinZb5oFdEpT/DI3QFA9CIGPjzWv85aH'
    'QggE9K2VBneSUHjAWPzmp/ev/3ozpRV69zmeanXvwgPQ5WlXYa6hpOGkY2SDTGSuEtCkj3BD/FlfPRM2uJ3QiNFzX2nDTfvK0TAm'
    'LH1g1Lu7hiO3NnhP/K7/2u6zh3/taSZCaKCFwYWnyggAhwyTnmRRdOLpJ6TZ0rhwkgeRzya3uyj6lQndWjNr7tARZF4/9yNSyN3c'
    'Yae4I3/g9MseIVd25joOMbE5qrC+yzOtK2gd5X1v47D8CAiThc34xqe043wz7DU5lt/Qq/nEF4QkX781HrokopRsePRh//pBr8Ig'
    'o8cXHnIny3F2XkPkONsDReZr6XRdKsXW+uz10qAEjtHsrpzwy3Ry4BbENrAPVlkGYMUmdaU7xKJykVDHQ2HUfH/qdOxHJGr9UErm'
    'g3ubRG0gKHRr9WhsZJq9o4K11nXketOWS2BVc2Qclw2bYqn2k0ftHI+0SS3DWYOoxkfqgbHD/p0Hu6/OnZbLpKc1Sx+PqppYOf2N'
    'vJw2e+4zc9qwCW6G3rC1//hpU9P+4+7+bGM24Pi4BRsL1alXlflQrz1DOkVhS8Xq1ArrxItstJ+58jyrr/x+xN5LKEXLcLabvSjn'
    'Miu4olHa1nye7PnN7XBOBwMVCWTmw9TBZR5cBppDsHqOwp3ykK2IqBCxuVwxKlrEWVBQpmB53L5ZjZeBKk/9LC/yPOXMkNjmcy+0'
    'TMpXT9VtBn1ElukGVFgVP2AGJCj1AKM5+zKmQCcPlkOlyHInxV5GeKt5wtRLRL8qC3vm1H/n7VxRo+h0aXoGLfyFYpEuJ0Eo9aas'
    'xd4xjbXBlqn4Gi5nytiauiWuLiGxhpziwP7YjQ5GVDlK253t6edtjhYB0txy6GoTUFrvcs3h0YdNTr1TVrj0Pq3JYB/yPLl6lGGF'
    'kt1Og0f/IS/uZKGmxBRrUT62QLHSBYN89KDRFOFj/NYVbcfVAccFqEI6ISOWtaD6UknN8ZydJqGzVJU6CcEDFkLCsgPCy4pPiqWH'
    'LJvqm0QSQeJkD5AbhWvni8DosvI03ttdYLY/w7TQjtOg5F44dGZ/hk7VEJPYRLezeWcUVQ2Q5jO572fduVH0rPSzPRN/5H4KJNFb'
    'TbWJPSQRFO0a7PcOSLbgD1MxOpbEQNJpP+ICC5L0o5M+7N73hkvqASJ8gg0qd520gL6V33WLr1/e3fzy+tObn61jCYslbo325ryq'
    'VqWpw7RvuSJBqM0oBbJbDiYG8D5HnIiL4RP5CogSyOR3hwcjHLiuLGedUMKAiZ80APAZuweEBnVTwntdrMBpVn6gt8JG7R0W8Yby'
    'hanvR3no+y65E/C7lIUfJoFS02bxx+Dh7W7BzzLZ/FRPdcxehOZ+YDaxrRR5pkWokMdAlemtxS1HSiXqNhY9U/PHQhiuQXFqPawD'
    '7nFqTIPTSymCRX4vqYw/uovupIJSud3IKaxPJTlnDVMgAlUU3Kmw+Xge1ftApiBJZCIM5FFj0yybAKcT8ebDP9QTeyKOrc5GTyVR'
    'dR+4yizneAqsmdIN0xwHAQU1VxqyGMyoZ9Vmr+4N7si6YhWXg3iF44Lc+rtcUwcZDAUh29xS2Ev6JfzPeoqt36PfXPNE7rjxNJJg'
    'gAq9Xsvlda14a7XOQFLgkSNx7rTPX+7JcOkSWeY0ITfLqyQyHU9u+ghVhDUhL7R2Uae2io7/EsmlPkZbBw1Ybuf/7vF0W4vxXIy8'
    'TrvcPeyCt3rfTbKJeRkXjbY1Hp66Z8NX7tnZpXt2jr8X5/h/4UKlEO2SAMV6+erMHY1eXbij8XCMj7MRPs7pCQtG44shfby0uNY4'
    'rB+1sHSeaSmt+t6FD16ArbfZ0pE/dw403BJNZISwX/vqG8MofJDA7qDG1A13U2rhDxISRcer6XkdxkNrLrDGdm4no9OhYeW5OPWa'
    'SwIkUARWVcKXVjiHVwA+uiHhXvfDvpp/TfpFUZVEMJRafISLBGDdBwDyUFrzGm21ImomWlMEzE3RZaWx4hSYWptDbprsSk17Xzyd'
    'YlOyrDRE3VrY63YwakQee+Iuyx+yupfdugRx925AODNAFHL+EjSN0B9ZJ92bEX0f0jggzLeOXZ1a8IGrexj2ucvb0H/u6fxqBW9f'
    'uMSL+fhqjYaX7ujs3B2RZ16c4j9cl33zyNAFD3075rTdxNhkpJlDZMzLOnZaHruk2wHKww5rx1bByLKnXPUEG7tdv7gtR2xZiBtd'
    '6S674cxmgQTR5g113YAcAzkN53eu8LlrpgcRENpEnPZoLStWLCz/O/RGvj7Wuz42uxLDs2bnb5x3dTpDtUuTK/WJSYG4bi7dojq6'
    'ebmje94WZjD3hPlT3eywkQXZu42OiP97d3/IysxUHSlnnsHXMFZ0KWYCpnsqmZO1QOQiXpG3caXRFIOBlxroMunKzhVnnNfXlLQP'
    'yKS7psls9pbRTZeta5qz0Ssi8MqwTWnj9yeP2lNNcntw7edybF+6ZLj2hRfDQJ/rWJ3s8J3z4E/CXDs0KWrg6foq8OgqrG2n/RQV'
    'aSb1afdKNF63I9M+PbuwmU0vkRBkecb36/oKiTAUL8cuC5sS6Zbzd0q4wTaHF7GNWowe8MS5A2q7tod1cFhLTNzDXvRHZ+uNwmjU'
    'qKNxBXIX7D1TO/M8VRR1kkCjESv06SpVN84OIm42xeBE1LfIBAP7F86A0CcSgm+HeRtXT6P6mpXfvj6ZU3w75vqclLdaA6DkN4g6'
    '29kF7srw8kd9Zd5tmnxHIbtWzMyhnsrtaC7+gExhfqCfIxfw+rco2Xa3p23yKgApclS6GiFn2HGsD4eWBCBgWU5LasqKJ+1OVTcL'
    'Ne2V+V4pDScq9kJhhw2uOCo859/O8UCp+xl6zl7/Wuvjidx7VUip9lNwqwmMYkewizdt8l2SB8k05fFMr7mV+7Nl0t7M53OQTgA8'
    'kIl8Oig49dHFSgfAvmSIOyAUPs0PP8xVusElIqcL6Ho19udaW2807YyZUBbi5vqXK/Hm56s3f78RdH169bZj5j1cZ2c9Rm+nHMPu'
    'LC/lxPysiIkYpgmx2gkxEaTvyof84AwiPPbEp+ZXIEkQp0IXSNZeCFv1jyC0g0+49HpWHyzPqF3S/LQDdmaQ1z+K+NL8KoKc/Bnt'
    'hOlUzc4cz/hUU+kMn9bniK7vMMP36Qra91lzvk/NBd830KA7Db3/AzIFUS53JgAA'
  )),
  'results/n7_counts.csv': ('e54df8b158490ee84f3932d449caec038681f979db50d1c953af4ed5de0e6369', (
    'H4sIAAAAAAACA2WcSbJsOY+c57GWGLABu01omHuQWZlqUNL+5Z+ziZv2W1pmvncInsMGBBwOMP7X95/8/ad8/6nff+L7T/v+07//'
    'jO///Pd//b//+7//+//8zyd/8zf9/edT/uNJ1ZPyryfxH0+ansS/nvT/eDL0pP/ryfSTP9/7LP1NT/+8/JOTn/W/jxi2npe/L8vl'
    'PGx/HzJ4tdR/dY/9sIy/D5mCWnr8fdj3w/Z3jJmJqGX+6+tzPxz/+jrToSXNvyuc7tO/nyrMiab419NynmpW5ffUW6Km0b/r9zTu'
    'U00tfo+ZmP5e9Joyf4/7fpzX/Nbxe8zk1Fa0DiXq7/k8z0Nz/PMWZqi2qhnm1X86k/bzoreX34w+lVmqsXb9236freU81yfL+s2p'
    'MlM1hgY/128FauznVc9i5N9zJqvG0OaM8mc8/Tzvld34zasyX7W2YJHjzxfmaUCFUvozBaas1o4G5fmbA2+goU1J1P77RjBptXYN'
    'qpSfLn2inAatREn9N+1g2modTSu+/nw84jRorUpffxp8AuM79W+U9qehn4ZUtcA/RfwEM1fr0tJGiz8fn7thau+i/X0VM6eV17T6'
    '510t7ZYlhQyW8jUwdVpR9tH+7HgrpyV11vLP9xuTpxmFGuvPZrW4LdLOUVf7tTB9mjXF74q/Lf20lC4zksqft7EANEeiOcWfwc3T'
    'VPWnxX9eC2tAc0Pjy+i/pp5OU0zMS+e83SaWgfbuEz7Ss4SfXk5Tk+lQLynLuk0sBO0D1RohXRu3KU5T11Q1lqHT+rqxFghMXrs4'
    'UncCn95vG/ZujOWR3kZb6I5J0NA1Vq/nbZy3sVqCnu01siqSKJlD0frUll51/ox0GpOWoUh5lk7Ba2RlBudCcyyjaKRt1NtYbmPC'
    'ro+pL7f+WlkebF2V1tUUiYOan7eJ2yqTWDSi8Z0z30aWiDVrmmNtHbcxrqZ/Rj+toflWaYYV4Y2YVRpYSGxYX7IPpaY3qHlbM9ZM'
    'uiPrOq4eDNZJImVKe0JGXWdyXBPymem0Dn1Mz7HM+S0GSoXPKatjxwbb0Fd5vrTcZq1VJLYg5nVmn8laSaTq6HxjDZkLnYW78TNO'
    'a5IKxKi8K6VyV2uyWpKpGraO0sJVpXzP2Idd2c1ailhJs/2dwM9kuSRSQ0ZIi+K1mf29e57mOjmm7G6uke9iTwOCib+QSkUrGMp5'
    'jcVnpdPctJI9cfDKqtfLfdDRr2S0ljKyqy1tlUb2epfbLG/c9W4pSinXm3wWa7bwMdroMYa2UsOL1xynGdWSGuBiV3rnhXOpddAu'
    'S7Fn6XI0kbQ4t7mf5qQNHzMnvbxoEW4zqyaZqDi+WTWW1ke+27nmaS4yujOFrHX8rP5nsWq23jqOSydRH9PLr7sQpEpHgE2ZK6Sr'
    'TW6rv/YNsKRIA9Ekr1W+c+hvT6JciV75RmgDViuzPQEjL4kF781p9bWN/fWxH1zCFQGkyDZr36ec4XwSBmUJALZtNI5YJqCt/ET6'
    'FUkbaVUOmUxN/83GgE2CDe+gU9f5nBT5WvwPAO2IFCCHPoSzm+1nbTQDi8g+2fRLEcAOM3p+IoaqFgmAjoAPOEgfyj/k6pVlvQfm'
    'VcsGstNRf1ufN5BFpIMDasUn4APmm9GGtRJstsZN9hRjNtJ4M9ogF5GJz5J6SE1llVDmK+LVBfPm7ZvQ5DIrDVekXxEOrj7UJwZe'
    'jvw3Fq+uBHvFjsgto5iyP+knMq9I4VD1lLVHmnSrbxs3VOYcctLzlC2RRdHqpqe0Gzf7qDLp0bI2R1/L+WmUQTSCUjP7Pv4c+nN6'
    'q2tEbZEB+pipASNlKefbxrKDBqGshEGVQdOfe5v5p5gljkhfyx9qWE6pZfzCEa9uAaiABLSB+l+bCTtyRfoVKeA52S/Zu7banw0w'
    'HkdQUBBHKJXElGlCP5F5RUDCxcurD0lf3uoapiMoS0NwoPkuWZ8U5RcopSsytFFFG63907HOF41LxKsrQTXhj6Noj7TP67mZbByP'
    'yFjysToASSBllcj9DdeQHkEpGr5gBGPRls63ukb3FtGy6UNgH8Tjz1i8unxfM5UuSZU4t/KH5WmDQb9l7Bi6AoLmWf95jZdXghOF'
    'kWslWJECKNb4ycwrg9sss7CDGXT1W2AHBUgujr9OUMFcyBRpei/qTFcmZXxFr/SSeWg/jXCogOSq9qhaWs7U0GF543HUYBkBBSKv'
    'yvYWLeHzLtkBBJKrN3yvENTgyAjGvFPpWMIyMog498VMu5ayvaMQOyqO79K64YVr+HS28bOd0a/MJBIULOfrTWHLzwI42LCk9icM'
    'bsBcMgcp/T42nxDGQrDblm/qYPwMqIMQiybssDwjBkNuu6afHXA8soUCoSFEFaBT7fJba8cmFtUxl5nQN2UpBRB0PH6sQHlCfWFu'
    'hNAlpHePnyF1xGLRbJdYKxOuackJ/T4XVyhpoXD8lRi76oU/q+04xqLaYOIyIXt0RdP7KZtDmi2EPeVz0g4pwJoPAmRHNxbNw256'
    'pYVGyaX9tsVxzhZqsn8RdjlFQLvF73NecUQLSyz8N9QoNFEUPzyiJF2hjKrK/g8WnrP24Fh2GGTRgk1saa0GPu5/bZUDoi2E0go+'
    'AItq9/l9Ql5xRIWscOz6s7yO/ErPbzEdJG2htj/HVIX2ZLjetjhasqiBtWAQi8mwx8/fOmzaQlP2S1CR3ZGV+AWhEtq8EMGCgfLo'
    'xPrAn9p/s5tPiD/Lk8rGablk0OPEixLyiiNq5W4MhklqhO6/Waj0hAT0wROzG/LNnssT8oojqv2SF1fQIs3spdpvjSNUntDMfE4m'
    'neWa+tqJRj/Z0ZZFg8GgtNo7gZ8FezGPUDwh9FPfqRvqJEBBud/zkiMb3Y44ONU665rf+Nb7rv6kBKBwRVGIr+UBhbr75eK86MgG'
    '2zcEImGoCMtyPmTeJzsq21LYqN7HxI1qjLXCFxwpLzuyJjcE0BaerAsiEIf0szsO07ZYXsSnJTUzMpOTVcclBL30CDciRcEVWejv'
    '1AhSOLw+YuWJNUEGhSIwHkOvZD6HLZKYlx/hru/JjyvkgZpYo2yTcubgUM5yrJQgwRqZgBQzCY12dtMhnYU7xg4frt1cGL/Erl6x'
    '/sSKXIHOu6IvQXRpEWb6J+ddQLrDVczaR3IMKSXXQq24o5tPjrUS+JKZUAAiR6kTOcud7OZNJTwAd5JhHWRcdd4we2uc9610Bbvm'
    'qe8Wr4oM/Qqfw3rkvBdrsyh8cNjACKh0YoFyPJUEyxOs0hCdqMloZasFIRYBxH2j9wNxqRIk2izmb+QgcbEC7WdpHCZuwcFc5Pcw'
    'qKnp0ArmyYYefXfAaPEJCMFxT1uACTLH1t/Vduy4JbVv+naDrZXH0IbA8rV56WdvC+KTg7ZwXs1chrRLZ2jMs+COJ7cg67OEqbCw'
    'ZQiS4hPTHFfSW4M8GBR3LtDbOKPyv2Eg1w7Lna7kNNqU/5l+52is2DjYU4Kb+QZGm4lLPVA2q5yJLZDfES1PtNgn6/+EBTWk6Cxg'
    'qrUeUXPkdFhjB6ABraB9F+rF6PfYxItE44liyzSADZdak4JA0ej5OqIm09VB4skapA9V9JTQtAEx6hXtTzSFWefR2aYmC9CB0oaT'
    'W9SkuzroXwYjW9wJZbSygrc4XiH7+9r5ZMOHonSYTZ3jMThTUoZ252VyHnRkxhbltNrjTYRBZWnzOSDFkatFZcUqsZr8EsizS7cE'
    'xF9oKVFvF+gthzMTstvacy2ITJHspSLoHeJLtDzRgqOS9iwUSDF6FaKBEOl3AN6uTambip+KHQZ0eV2OH6WQ7SyBQ9st27H/8jLN'
    'uqvYVgZMI3hK6BCXDgrhmHotAG5em8yOCVn3O6/+RE0HlKljvzz0QmJA+DhtTuFTHPOWGyf5RCUdNq1DsWWRLuZ5ROcT5RluyR+R'
    'NevAzhlj3gyQ90tCpZoEkCmGlNb3ZfAmvKCR+s4LpSsr3AdPqwCUGFJWqBE06ijVM7GdWSIGrc5y6BxM/tZBuqYPNOoz2p1vsqxZ'
    'wCqoyjZI1RrmRHNo42zDyUJBsa7tWmM4q6Y1TY5nc2v3vfFkzTyH7CP7oDOYHAY1ae9Z3J2wgk6LoF+TR4cQTSEjIyMrlYy7ujuL'
    'ZVmoMMHungiC25KHnD7R9SruTm05hzU34a5OwPQODUP0J1N+l3c+2YElkb2UykpEMd0AHwpI1TtebxvJLsMGGeFBOkWqpg0sjmPi'
    'WqWdF7OsY73GZAgqMclQK3MeVkqy3rdq5hvD0dGXCqk8hYWGnHrt773lyTZTLJkPsGbDkxVMTeucCQfg9JBiofSyXRWiWyF4gWRo'
    'OlX5JhrjyZoRk9uGCBMS1SmV615zlnLH631Tj9J9KLQVmCIBIh1DKWMbpmO2bH+yDUMpvIPHrDA+hLaCaXWePXaMTg+drXDqQkgN'
    'NjW0zdX4V2foKKWDdQt3bIv8/XDmS+YDZf5q4dY98Q7a6SF7aNKpdcJ8gQFtnBRNC8ep3rnV9GQNfWZyuKzxJjOgGhNW+gh750B5'
    'EwZJrlKzdF6uBJiJHMjhmT/F4fzOfkDdy/xpu7UUEyvWoHdnuUfZYf02Rs2WvBfSBwofZCGWCbMx59lnx/dbuO5RpFXNOGvsGo7A'
    'IzBmy+4ksSns8NGVrRZelB6l4iSRzte6I+5XeO4Uk5A5aUf53mR2Ys6hfTzC3r0gj0kQJs8lwyrTPwnDAFqtVkkf4fmEIdrlOXXc'
    '9GYhm7COK/4e5c7P2xdEmcnOQHaAJEJpCl/l4Suh4bhJ8HSEpbkc68SYCbB1XkzVNRTuaJyZAbrI/Sdzd3KdcMNLvk54sMBu9LMa'
    'Zgi2sLncnI2rhEe06XhZAUkI9S3sDWykWhwR6fAMuGvsC2SUPi0ndiZoxmALT4ywPAuGO5hBharV+ert7KCZA7rUjK7JKmlPCOZk'
    'VzEl0loF2WedzSBsYcIkDQMF+uLrBe6x0njfc6TMJNClwr/TVnC3BlLQZES7+e6gGQULO8UoUyeYorPf4RqLz0A6nK6EvYPqooUD'
    'e4TUTl4Fs6R4ZWA5G4H2rldIT7hi8KU2PRP3wmiYAxSsyOegmGmgi1Sy7EyegMSXFEqZThE3b84WLle4kIiA/SNib1PuVm4Eiwvn'
    'cYS9gxBmGEy2R3ajgqUJERqKPdMbRjxhQQOGIY8gYbh8qSeUuUzY8TlmIuhSY/OrxJMKQ/WXgd9XT54d4X6F9Qrcic60DkEnlyV7'
    'gYOqJyH0KX0XdUAEmi2V09n5+CGYhUom7dNMd9DzSVeUXeYIXrjLCTWSpC01oPER9hZ2UnCgpgIQJj4VWpNxUMio0GleI2rKwsKy'
    'JZPhCwUqOhVgcVwOSK9x/ZS5C/oI29g8ypZrWxQq6qhoheWHpZD31eUJhymsJF9FAlhOHYuhTxFsHmHv4aAsiMibRZA/ExTrYC3D'
    'RE7arYiJK63DBL7pJNwJrYV7ZKWIFMcbtDdRXaS/xdnYRcpkyvkNTJqCEclf4f6EUQvZ6QXjIPuoz+FOZUziYVlTHfSpACzgSoMs'
    'BTFkyDbYQMVLR0FMeWxpYhjY3U6MLc8zSZgrgMr1unkzH3SpThBA22HAF4AOP4BtlGc5AzEBsqU7yXzNkCyQEDsnU3NcmsGNV0yD'
    '0KVOF/0QIEE1tAFn7+qLRW7kSJcnDYMpkzpIsAvZ9xaudyAPclGaSRH6SMWyTY6coOL/hTJ2o0qwzbHqpka2NPhGk29YG2mivITp'
    'TR3ccp2cGRL6VL0S4lSh4Y5OZeaZgZwx7OYtj+pPHBOhmQka6eXaGq0SYJ5cyFUp8yX0kXWDV9cbF6GlnKMCWmIhhczjZOAlPo94'
    'dZ6Qgz1QNDl0xQUkWhpZ03KO+9xFWRMkAwkLUuIVJN+dPIcfqWudVVzpiVdTF8KvbDFwAe8M2OnYyyPuDV0mzJkv+BIbAuqd9oOC'
    'mpHv/ptRsTgngfSAzKSDbBATQxU0Tc/1mlehkxYOLyLj7hwYC6J42NHQYDxHPJ64zr0GA9UIGtN/nfXTl0G1R3PNsdCJ1BX5jNoD'
    '41g4TKQ2cMeo8xHvT5zUokyLTAhT1SwqtIcgpMKhJ+5ddVp+OcmhQNdpIAgAgp8mTJrLdQ4mXbZ4p9RDC8I5oLhJX1qO0AToLrw2'
    '80IntTqnsbLxorZLyKFR3qaFjM3bf5zpPuIVFDKICJz9TrM2bM4k5DpHqaZdmJfAWqa2h0zZIK5scsE4A+nxrPWW95UrXtE/EiwD'
    'BRbc0z+bMxHiOExENRlDp3BBgdQsO0TQmYAjTAITcDgHxFcTMls8ePvU0nTnFWX6nPUNgNsh7apJGTqFE/xS92yOOTAVpGuCoeQ3'
    '9v7EE29fIecALabhFgpJcDdxynI+1eQMneQknCZSWFhcAEaIDOBMQzudr/h84p1quGRMBXKtmGsBhDz5Yz3iriakGqhTrCFFC/x4'
    'lqLJLGU4Vx36mc42mafZ4nw6oCgYuyYg80vw0jWZU1n6qeZq6IQhT+ilTqs55NAegERwiv0QK9V8jcW7p1oU26B2Au4jky8TNiSA'
    'aUfcu8qoB9VIUoVCWjB3Zk4r0WKLE7xU0zZbvJmL1lmrTolBB/VdWgQzdcS9q5laThJlArMZxCZfIc9ONnIJLKQLw+1Kjji5cH1V'
    'LhwSbxUyUuTVJ4T1XRnvKsVQk2o3bRYQ2ISX8CXVoxPnO+7KzCdO2YD0qew4u60FxpCDR7VjHnHvqjrF6ntvayXE0jsVasmEEziV'
    'ks+umszZ4lSvSLczWTPKu2RStdzYjXYKFCXuXXVGy3HXIDM0oAup0AFUFKiSfOKIalJny6ddseMUjqKaAbgjSzDIYpzBl11zW8gA'
    'ocGUT7TNznZn/iFUUz6UaC3xxMktBcQujnnCCxLLQMbLEh7kWk3v0KuhV0x5AtcFtzQgqEO57CGkVc9OmeLZ8uYMBczgj/X+itcr'
    'X3lOxU/XdpjloZMwLvm8RFiXTIHKhOLNWiMXc8LvaqbH8pnCNqrbJpHmIjEfUmIByJ5PqePHqXbiBL2eGr4GYJB1K2bzXC4j4Cal'
    'mOd8m/DZ8tQ1asdIvX1dyaRwx0UCk9D6DH8XSEOj1uBICJgT/VNorHASCEOhhOzLfX958rCBlI4Mx7fwYQuckR1qH7xed0E1jG5g'
    'EaSXCoTLDhfbzCaBNfFydWcXWiOvDVQcJ0swXcs5dcBdBAo2HCdBL3nvrno185pOsDla45DDhmlZwRcHfNRdmG15zoVmQY2S5rvw'
    'EfiNDl1f33p6ewnGdeCoWpNmmHmVwRrhuiAZqscr113GbXnyRI3IC0ALrTtMsss1axVOOq7u6m5G0l08JRe8wLjwxYOKyWJvtMZZ'
    'n130jTzVCgTyiWKVollL5ZuJBZDzVTdTQ/QitCQ1GghAHeqUDNP6suR5XdNgdmjLU6jUKFBhPHK7+EY4YBnPOU9eqpohopferFPS'
    'yOc4iFvk608uR0DtrqdJoi3PnqM8ICKBW/k5s9ADsijuesYusNdMJ2Rco7rQ1Fzl7cIJhNeVFMOR71deDpXaeEgZyjCF/ufJRnTQ'
    '+5X3/gYjh+tQBNxIWurl2GMIw7DOntDW5ShH3udLYDIgVhTJkbsZJi7lma5+mjSqJkZAUXC1C+sjH9OpjjHbTsrtvN+80ZaHFtLL'
    'tHiW0oGOzcrIQZXrz00d0Ys0LRlwPrMrixpnGKikiG0eAruaPUKeDaqkpJ3rl2UWEIWvKZjTfErQP9UEEr3kVyBhZEMSrOSi9oPi'
    'IiE5KTHAb8vHk+8QIIWwdTqnJmjdHJAprL65sGoaiV6cPm4mCGSaJJfNnb4rok2givjop5mkLc8powaQiiN4bMKGgWZoPdsJS6vJ'
    'JHoJfbA+6rDM1bmgEnpSTpqjdscznzyJto6lRH+0WFqnYko+swp3vt5f9eq+7NCpml87ZCpOQCoux/FcuGZWyfKFeEMAx6WDlXTh'
    'pJSZ0kBp0gkiq4klevU6XGdIwhviWh5XCAzeVh5k3TxxNbe05QOtGGXnh2Wisq8+yLbKVv/kvb/q1QkYKS8kn4YGCh0triIJJ8gE'
    '3/NlhmnLU8HRiYAzHkdIxidNuqUP1nX8i0kmemlnqUrQEQ/fxWG5QKNBsVh959E8k+Wl9owHqEaQSJ3Xar46os/jkbf8vk/TqZf0'
    'HRyiSQgOebzpAnFZ6Ua248jPJ98pVEjy5is7TGTPoKuJ4C9RXc030at3ylyEO+EVYCJkh6gc1ro1MzTnek968ugn4aHr9oNoC8sV'
    'UFYy1Meem3SiVx9YXamPkDvxEFlpxXzU/lR4+WMfzDtteYIJ6gRIQFWIOJKmX9dbzXT319QTvfRsUKiSdqEUlaNOKRFHacuuvzb5'
    'ZPlB0kNGvRHvaB8FV0kKU1z8amA/1fwTvYQSKCzVBKi0g80VkhT6VbBILH/PrymoLU/JF56Zqlw9JxilHJXczXz2xCQUvfoiAyFN'
    '5A9ERUlWBwsDeMg3hK6mobY8lMrQBsjBmLDZuddOtDVugszF27BQ0gTX4GDXKnQmhWtUY3fhIYKpMx5zUVs+6ZxQ17M478KoQk6g'
    'RuqEtedHP01HGd9TZwDVRm5VWr+glXU6BJb0gXTRqgkpyycolwFNT82d/MCcIAoKzce8xQzVlBS9CAk7hf9y9I0IVEAIAkSKKt04'
    'Ne4fl0wfeVkojYdC4WZ+QpGQ5qttgJ292N+0FL24KkP2WFFB8j0t8wZcmFpAoBujmZfa8lA8gEl8s+LFrkm6gLkMqc+1h2am6DUw'
    'aYQ1irWkZdorLlBQSNFWOCI8HebrICv5nbhaaHB2oBE3DNLYt4z440oq7kx8wYzUEmmCXPDTzhHGhOD8Iu44rHhd6coTbWtAslcO'
    '2LQs0pD1VXAnINTuBpieohcpc+5OyP4McD1XD2ARqH0RiL0xr/mpLU9EN7v0jJRTJ/UDHNGWaAbngtqnmqCi13BZl7yX67ng38Eq'
    'yzfJYPWvxzZFtTtgp2W8NWdWNFGcSmGzNlpaegNZc1T0klcmykVPp6MlQd3AHmhk0OqneqSapdodqESdivGBsfAI1OYQAlSX494z'
    'YJ6KbjqBlAUJoDo1WrCOmz6rLua4RtpMlTtIWxQR6eOAgiBN0ig7WAP+5tzq+lRTVfQiFcJtoATcFvjulIS50HAosiGzcW5OptvB'
    'Nm0VyiynF7XLqg9XBFObdNxMpH2lEgPkdI5sV7OxlZKbCczgvtRuiU2k8jogqDjb5A/gXEeTGFZ/6sbIp4OvYKrbsNpzfYVSNjYd'
    'kh6+qNolH67FpUungy8hEm8sx3GUNe5S71zIDs3TwVc2E1cz+S8AjsoQxREoAyyVDnTFTZfTob8OSWoo3GQMSybzlD9kV7Zd8BSm'
    'rug2udHyXRRxF0AyNSPRXdhM/XF9HebtwCGkoFvqDSEmNLVI1SiQnCQZ1t0HXwnlHJddDkaBpcszFSyVcsoaddDPRdlPmMFyD6qc'
    'NKbe2RZToxDIJF/kgUo+CD/MYdFNx599UnRh/r0TYFWnwWUCBND6ocvDNNbu4ZKbBOnikEWRxi7tUoQlyDjr7eHNpjwPSpiqcfll'
    '4L6ssAJ2QJmsH7TKuD3i9XCMTLgHiaqQtXOJM1y7OPWSc+rCfBYNk1SsuX4ZExwgCRAXpw4dak3y+OYwpbV7uFg9yb4DGii6VVjG'
    '1DR9jnu63/CG43ea0x2yc+vATir+gMuQMVLbdbdj3h6Bd5JtwaNrA4nQpFXWQuqObv2Xi+ggMWW3XCelARQyjfAfkMyapg4IepDP'
    'Wpnecg8zDBoVZboU8midp8swtBCyV+3uoBku3/k200dZPfEdoEPYMahpoIJirYvDwhzX7uHbXXYiXt0KRiSM4J4c9aP19Nh3rLHI'
    'ROLZ6Z9wBW6lBtwXqioZ6XF7xOsBkZt9b0A9+uBSrKG1NFEGqF8tMddFv+kCzMw1LKDggNUx3GDppFY3XeIFPz1sZKRz1HdywZDA'
    'JHxTVpClXOMcJrzop612UavwJgXAyAh8b8piUCtwtcSU1+7h2xbZmRyKpzWLRV1SIX9CCcb9hve8cHXFJLGLXjGmMlWjO0xd5OdW'
    'ObjPF9/cA+6VMkVSx15dCp9IzkFcgF1PZOUbc1/6EdAls4Ymfgd+MJxtgQto49zh/oS5L/dI03fM5QCxM5o5Sgstp67Ok56Zm/2i'
    '3wKjkl8tmbrewS1pAy+SS3JEI+6o4vaAjWVU3F3gG2S9yCNCCGmL+qGKwwwY/bQY05eJOOrJFjL7omfl+gwhxJ1Hfz0a8bR0O6je'
    'YmUbdUyUagzun6zbw3uufpwb12AOyqwwkfAUAOBEZVA991HVY74eu2qzkI5u+M/BtkFR6OjD3d8d9J5zXSlcHUXVAGeVO3mAJDPW'
    'k+Kfg93DXNju4cReYds49ik7oRLkoxcV1Xfm+5cRwg7cayVLh14JhFcSJ766Sx3hPFW1sX8ywT18K6fw0wEDbyKElzjnUA9yaONk'
    'FGP/loKvQpmtoEaJxMjkFtMakBuUgiowT3dU8XpQREklaA+wIWV3yekxbtuSfDjW5/z6gnZwuFKscg1tgJQKxXAEwI3SwXIrueL8'
    'LEMYWPjmGtXBEzO8SJQPsg2oy6XlY/9eA1n56euH5CCF4KU0yzlAV/cH9wEOdI79Qw702Bf9ierlYhziBOXo1PGRdKwn6R37Fx6c'
    'Qrc7r5w2eb/Fpa5B8EeppwKec8/+E/uXH+gxQQqQQCyvjn0jAhsOPijp6xcAmCGj31o7fzmaa0+kfnPWnW3V0RLqOpxRmCPbPZrv'
    'uzTCLzL3qUJOmqCQO6yX9XVNy9f9ku92ZZMVjTvPBG2UNAW1/TKPV01MlJ0u4cznyL70JnisA+KMqQ4kxPQ462uuzB2T71jJsAUV'
    'S8yMWqbKBetKQXC922667HQx96FNzJgOLRdogVukQjVcI7jI1YyZO2or/JVKRely9TvHnlzj2NWW6faZvz4uyQqEZRa0QtJBzG9Q'
    '/qDdOkRDmDhzR50iVx8LU9utQqhWM/AN8oGKvLMxJs9On31TRxoJ6b0me0/tX1C0LB2eZ83Mn238E7F/DKP4UokLAjgulaTK5Or6'
    '9SXm0E4fNF5DSyyrkR1ZiOVYBOb23HH/+PrC1z2lS9XXg7g7231DPPcK+ShoTn4zjzudeH32bQK1dm7hGN0tasj44QWBzjpORYFx'
    'wNc9Uy+nvD9sZ+TyVvjHCKhgEKpL7fbpr08DZmeKOAycQCh9Z4nJwuxKt91n/1qL76Ls20Lgh+oLAFSvZFMdmgwF2nfd5uvTp+9k'
    'QhX5PouCEOHUAe3P9XLKWE8fq4GR4DS8a82lA84jCe5Xk+xgu7Xm/SGZ9PoM5+kplKQMlTsPOqCQC9yrV/CRLiY0z+aeifSCf/4k'
    'nP3KQCO4b2EefLGU/eiOubbdZ3IZU2OjYoW4h7LsRK2rDrmvgN35mG9zT03dV9iXc1h4cgFvcgq+cs4BPxlHx9i3Ty+7jCg2DpXZ'
    'nK6x0uLZyu8rNp8w7+aeQCdu04MC277FCH5ZfEeBYCIPefr0X5/WPbadWeI2JitH4aQMV8q+C7HHZj0Y1q/Z9/0tiGffcSxk+SGp'
    'ZBezf7dEQFl95q9Pc0qNQBF9M0lMakI6OqmCqCf7/v8BpKWEMTlMAAA='
  )),
  'results/n7_scan/noroom_off0.log': ('f21c3d07efb7c4a8f52d236e6b5eb8166e97ff772af67e630dc5bf94e118d32b', (
    'H4sIAAAAAAACA32US27TYBSF51nFXUBk3fcDKQtAQu0A2AEdVIJ20DJC7J3zu5CUiQdW5MT5fHy/c313cRl6uhS9PH//+fr4/PRy'
    'YXp6/vbw8oGEXJo6hZuEx1XDKZulRCNIZky8VYSYXh9/PFwmt5zT3cXVDpiq1FXNJKKAeVBJWwPnpFHG+L7sH1PEt7YdWkfQpB7P'
    'BU1OaaVCCpPCD2pdyhqcNyhvmQtqcgA1plk8QMfb0qkq3X0kSDMcaVlvSZW30h0aR1Cn0W4nURNOZWpllhYOMhA9ov0Gdd2qduiR'
    'J2taTw9PyKWKaxunGZFCJo3hhts7KP+F+pEoV5q0SBI4waSEuhFwCibMO73U+QYN3WYX5UeiPGlaFI9vNsYQNTpiq1OwOA56mF6h'
    'VVvLgsaRqICoGYYoq+jmosFHG3iActgExnOD9ta+Q49EoeyCri+qo6WNc9wEjyyYNiZXaIZZXanDm8ROPTIVa5cs8LRggottEPY1'
    'Z2ujiDC2qltYld6KFzaPXOXChBlkOeoVyxq3SZqxUxjmPiP1Dlub7WnzyFYuDNYTMwBcE+uytpNjV4guYBUk6voGwMptuvuqI1/F'
    '64XCjglHrWYubOPPjG5QTjO6EX2tgWZu0qfP95++fvl4f0eLH/TLuM4Y4dlDz4iKY85ehiN+n96u+T+DvM+AmyME9k9SsgQdElVe'
    '6z1GmZNQzzq49i1DzMZ1+gP37hVuMgUAAA=='
  )),
  'results/n7_scan/noroom_off1.log': ('b52f738d6658eb6ac1009bae0f9cf5eaff66a9cca2d71f31df7ab77c263e5c1f', (
    'H4sIAAAAAAACA32USW5bMRBE9z5Fn4DoeQigK/gU8cJAYi3s3D/Fj0TyiiuB/MBTs16RrzdXpo9b0ef915+v9/vH5w3r+8+3zx8k'
    '5NLU0SIkXMVqQRns2WpDMlY+EirE9PX+++02tlxfXsH0A1OVujyDRLiZHcwCK21v9Uy3YOPB9JV9MfvETOoRTzBdpN2pGERsCanw'
    '6JiL/mcK81LbUNMD1JiGOxjQSlCLylXSgSZNd50KyQdUag1f0DxBnUYjMKnyeAgmzUaIOUkaYOfY2HdoxYb6SZM1DQKCE3Uo2dZY'
    'nL3aSSsRaac/oZpL64KePLnSJFsBCiOOodr3gEiPDF/DmNMfUO/VV6Z+EgVFUzUolMFyFaCFoaVg3HTw67D2hOaKCxonUVA0E9mA'
    'Bk4dRiNsPQz7lt7SNvk8fuoSv6AnUeEovdhFnT3s0ERJd2TtxE0ZKcSD2rnmCjVPpgI4NinA0S/AgqZHu9RB1ZpQ7D6b2r44L+pJ'
    'VSqojgxARZEm9+yKU3viUnlWW1jlc1jcKrtqlSdZiZZyNpojgUs0vdfNbamlFIVsxRrN/IdVndWysXXSVbwx6DmwwVCDTERsPzDo'
    'XOCv1HDeb1i8Ade7UidhO1Ph2EMHioTLiXV0V8MbxcTgTjjXA2u1jF/+AuvVx6YDBQAA'
  )),
  'results/n7_scan/noroom_off2.log': ('02c3068945f95363f798d48d4180805d729917821694211549cfdc5beb0a2856', (
    'H4sIAAAAAAACA32UTWrcQBCF9z5FHWAQ9f8TmAMEgr1IcoN4YUjshZ1VyN3zWnZmHAhi6JGEWh+v6r3q27Or0OO56Pnp+8+Xh6fH'
    '5zPT49O3++cPJOTK1DnOJCIcHUU5ETHuSsqjNRPtxPTy8OP+LCyb1M0toHEEdepOXVBwmI1KW800g1QD711krtDYTHfoHEGbhk0U'
    '0OpIZqqsqGxx0glPHW25QLW27gU1O4Ca0ihLkih+DVE15iblUFrNNl3e76EhO7SOoEljtXqqXlas1NoxGtNkDGwZG1+g5lv7gvqR'
    'UcChSFgh2i6r/C4BLUGyahQRbdeeRm9jO/TIKHea0qXUlFXL0GLVzA4nixErVZR6gdaWs0OPjEK/Zpgb0KglEN2AdHSiyKYys1iu'
    '5Re/RSqOjAr4DiYvqROIqNB0S7cist6Je2Ts4hSq+Us9cipgPBQuKnKCoA+eZZZpnOQllazBcRE7vcnuVR55leDxSj6whViuXnCi'
    'Ul3diOXg+pOr2t46d+yRW+kLo+sSGCxb6kUwq2MYjkQ6G7XERa26br6HII/8yqWuZYmMsBKMNua2nRFiqJ1WDOv0xTG13nwfrTpy'
    'rJZjA7+BnVAu8GUCZ4qIUSYv69wuZ4tGbhY3n+8+ff3y8e6WFr/ol3GfXOrkESfUcfJyrMSq3//ZjA2Cl4FNiQ9LsOx18+uefwXr'
    'e8F7O3u1N63RgDVrgakQRQIKh08LbpP0TfDEGoc/0cBhu2IFAAA='
  )),
  'results/n7_scan/noroom_off3.log': ('998ffcc369ba61ee093d7a6837a09cafc74e1127be9b3b5ac0a941e895552dab', (
    'H4sIAAAAAAACA32UPW4UURCE8z1FH2A16v8fpD0AErID4AY4QAI7sIkQd6fe2OyaZILVSD2rb+pVVb+7i6vS46Xo+enHr5fvT4/P'
    'F6bHp28Pzx9IyJWpMzVIeLjLijKsLC2apNm9LNyI6eX7z4dLzzZ1ugMzj5hO3cZgivpoKOVItrMzKSblahi+MYVjM1lQ4yNoU8+M'
    'ABoV7kElIS6cQyplgwNk36C2me9QP4CaEpSNAjqipkXlyTBAoNQabyJrrlDRLXqH9hE0aWwZK6plLlDaXTragE50SkbKFWqy6Q71'
    'o5xg3YTEgqbAQriBB/PKSbvHuPCVK1QR1Cv0KCh3mhw2QKcyHVCfaUsuMklpYeUb1HXzHRpHQYEynQ2oGaSaILconL6ErB0jzq4r'
    'NG1j3qFHQaEuwuxRoJbMLDvU4WmUk1W2Yli3TmVsnTv1KKlIUFVX1VEkECA9TdVWqTCJdYS4RdW81a41j6JKhISqNyTDKchdXxEV'
    'Q3+LfExTx8b/YZV1m30B8iisdGACSwrs6lOsncV+wtAM8g4PQ4XfYWXzWdg6igs7I+gOCimhA/LCdvWwSFGomaMEcjUBu7axnD7f'
    'f/r65eP9HS2+029jPTvPGYLOHnhm4JdnvPxzev3P/xrknYZa6Q5cgYZCrZGuIJFqxjpSaonBJJgnbxo8N9/voTqKt2A8bovV7xRB'
    'C9eNhJ7brAsP9U7XsOHr0dw35dNf/Qq5dDMFAAA='
  )),
  'verify/g3verify_rs/Cargo.toml': ('13cce447b589eca76f03cac868a1377d9a308c5804c201ef4f1a592b70e25b07', (
    'H4sIAAAAAAACAy2MwQrDIBBE7/sVwXvFJOd+SchhG0dZulXRNNC/7xZ6mjcPZrbGx5Mzdir8wnSfXF4vdEkfR5ZDavnJ4GcfHCHK'
    '+TdLWGZHtEU0lIhyCMZuvfWaROE7FDzst7bzprigtlop4vHORol1gL4SeLRrfgAAAA=='
  )),
  'verify/g3verify_rs/Cargo.lock': ('05034f369191eb306c9c5033e0e6383fc6c275afbaa5fa43a3e6561ef804b2ab', (
    'H4sIAAAAAAACA03KMQvCMBCG4f1+xZHuodJZEJzc3UqHs73Ew+Qi6VXIvzduTi983zPg/Sk7BkmMvXRYyWSyUkoNL5GVKxlv+Gh4'
    'pRqLhwFv9qNaetRYt36HUjGTHpSQNzHR6OHDdZeieMYJYJ7ftL4o8rKAUua+ujh1IqG5P+pGf/Kjgy8BkCbnmAAAAA=='
  )),
  'verify/g3verify_rs/.cargo/config.toml': ('8ea425180afe0e2fc0cd930d9a5dbd4da07aebe14263eeec4d6df9c7ab996a20', (
    'H4sIAAAAAAACA4tOKs3MSYnlKiotLknLSUwvVrBViFbSdVbSUVAqSSxKTy3RTS4otc1LLMksS1WK5QIA9Q1fbDAAAAA='
  )),
  'verify/g3verify_rs/src/main.rs': ('8d3d542d9b618de7f07e198b043ad9e5b54a6e019b3debd37b8fe9f88bf18557', (
    'H4sIAAAAAAACA71afW/aSBr/P59ikpUiuwEHSDe7S0Kk1bY6rVTR1XavdxLikLGH4NbYrscGsgnf/X7PM+M3MEl7e3dVA9ieed7f'
    'x5eXp+L+ai3TYPEgul0RRL5MJD6iTKSyG6ySUK5w4WZBHIl4IbKlFHK7dHOVBWsplHRTbykWcSruZ1dWZJ9cAqL1NvVjJX5L4zm2'
    'i+9+7P/QEe/f/vpB/Hz1008/DHp2R2zSIMtkBAiJm7qZDB/EIo1XQqXe5f2Vhut44tLcWLgqGzieQ/AZx8/+KlAqAAKAyIbiZzES'
    'j+6s7zjuLNqJJFYBUxhEmbyXqRIqB6HZ0s3wazULhIc/F3+nI9FjBiTE8CCiOPpTpjHj8LBbPHYHnW6/0+v0O4PdvyIhLPklD9Zu'
    'CLGAaBLIFW4DqDKQpYbcEZL3Fzs7wk2l8ANILvIyu2LlTbBYyFRGnlRaBgTzF5Gk8X3qAqrlyzCYS5JShymttORJe8gwhHhVKCNO'
    'fZkOGcjK3QarfCXGYhGkKuvQzYifpHLlBlEQ3QupNayI1jdvf/n97c8ffh3/TUO5KWFHsThL43h1BrJy2jc08nIrPXzuqnwOZZCd'
    'TKCI8VTL24thQEGkQEagxDqAZqTfESomSgwGIRKZdsFWtsT6nOj5OPtsjW0WmudCLYHnhsKqWyjw1Hm2K3L9UqRkHVBMFqfSFy5+'
    'PaxWMksDT2wAKt4oMemmLsjBxxT3gB+/hLeMFUQFokQIRsMKdGG4ENfv8AK9JQnBoNiADBHGcQKdQWC/vf3nHx2SnMYE/CGsWMOr'
    'qf99niU5bgeRZHzjoRBn49Ht+E5Eo9voTnwcQhZ9/A2E4zj4hrXFYU4uqUa3uLw7Y0BJmCsRA8rZh/fv/v7Hr+/HZxXUmp6iLimJ'
    '6S4NpCInV+69HFZRIRLjMBbjZSAmKpPJVEzixQIA8GMF4LCeGeCvgjCU01KZ+GftPxV3cLRUKhI+VFLaXQwrSMlSgElksbgbiQWE'
    'mFrjV8nqst/r9ewb4aWxUl1vKb3PxGP4YJ+c5ArazfzhUEbrm+oqiIfDf0BJ8ubkBMhyLxMftJU8nhBh0RA8Bn/KDl/Ng/sZ7gTX'
    'r/X15SVuZWQ0Q/azLeKKDiYIgBcwuYWwYEZYox+DNro3YquBQbJ2wagVuuk9mKX7Qa5sBo6VNVTfTdwwjDdwb9efebEv7SnfJzNS'
    'DSIZKG59lN4t/eXXr+/u9CM/SLMH/cTSO/RG+65DvLBJpm50L0vi1CQ0jrlyH8RcliGPwIFaDQxUAoIwItFMiEhKn9zIGLF4J6x3'
    'hasU+tSc0pWqQeKb2rH13by8C1uusRZUrBkD0hLTdEBgRwxnRb4IZmA9Zp+w+lBLlazsk93JCWW0pjksIlKmda5kuOiIcB0WkhcJ'
    'Y7ZF907M4zg067U+MvEFwK0EJkEbHWjWpvDCW2/KhRY/LMS+DqeTL+LuTlxP6dP6Is7F9ZVt4wukjkSf94FKLfVLUW286E+BrwYI'
    'iJFZxEXXpY+Bu+tAvCsEEuhHbl2PchMsth7cLDLRkaYXPw1Yu1Oi88IgSbA/48gs4DtwqTxJ4jQDoMErpLeO4K/pjSCTgS6U3EAL'
    '0ilkKbeJG/nW+Qoh7VCkrlEmQBjhNoWaEn3kxN4qGQ6hSKtOLXjuT0EB5bp8Zd80tobxjJwSSummL2iFli+DYvnXrAbwDclf4yAN'
    'HkCj5wbo4XMLVcwMhQ5SsK8y/mUXqtBKdVQSBtnMzWaQnGWY3WMRQIbifALXIWM4L2CyPTRXAglWkg7K1XRR4J70auuheNblBiGB'
    'cjrrPknlOojJ5RGXyOGzJXK39vpQLpB780wFeESLI7kxOa7JtB/G4HcZlKxysCo1WZHARQ1lVOxwnKYBkOgZCgukZi8c/VQ2CYi9'
    'XgVsdwi2BtHdWqwsAOsQOttxRgT8W+C28EKGpCkl6HZDum6aug+KC5nE9SmAcu5FpBqML69fY//AaIACP3mtdBGeSLxcJVHQU2FA'
    '5eFcImEwJHDFYd2pYyKySYKjJRMPAxFP9Hl76+ofd3ducWdQ3hoghPhUN8D2F1QzzwHZW3YXqZQaMehWSqoKl0Y02zCq2WbqeHHy'
    'MKPidcYbLDLOxgq7qe0VMYAacTRgb9sTPxv7Ehys4Opui0+WJrZRHTFXZGAWNpDjdfgXB1beeDWwD/eheNfuuhFd7dxQ4uEyv+Y4'
    'DW6aSyH5LN64yLTL4J5yUlEwIK+h4MnDjOuFxKhE/+6CwwOEedIzns3osGijHKf8CSpB+R72YAEJUPro7UmxkLXlrzuA2bPZwxxU'
    'RikHGdv5M0gsoORblm237Kd/r/y1eALpvZuDx7vGnR3ngxYomrX+AWv46NfZw2UriyUnJSv02X+RIXPd/zoGLdUTt7dsTk+46HOO'
    'hn926Zb9IvNHTAIO+5UWcdFmEX7UtIiLyiIu/kcWAZT/F4vwo/4Ba+SHTfb+CxZRY8hcf4NFwAgqi4B5/CcWsasHaS9E+clNRplE'
    'J7qw6JgCYkpccAW2iVF65pHvotPm7LBflaxc9XnGsMrqhEJfRRpsobHs9NAqiuA2FecjcWpZfZQMxGl9n02+abclQ1P+fJaSrNky'
    'JZCOv42gCkqKdbcCGA6J4NBKRJQkmA2MfR856uSyctXSXDQKZBQtpdehAuDii78ukPnRSBdX11fCouxH8kXzFLlhcB8hSadoy+yi'
    'rKWHKM7a2gSGUrUKeYO1slPgVV9RaVooIeacz74U6ezLsWxWpqj9LqMh9LmJBMKfQLqFRyIG0CVZNxs3rnQhU5k4vGO335G8KycY'
    'XMTAgNFFwpUCSE3ROOnN7DHq9nd6/mThp91FrWN6NuTyJbj4pHs15GDX82SClqWEb+mpUNFU0qAi0LMCNek5DuBBsZHfUPRIIx3s'
    'aE5ALaYZTW3JjSZQtotdJYZqDrhlSAN8rePALyivwB20WGWTQ3OcGZPwfKOz395ERSEcNZVIMOaukrypWGPa2GrlZhmEkteJ2xFE'
    'Sdny8Uhp06yhqeWi9bTXPqhdEEfnAerSaLI1a0imfbIduvHplcuzQY63Kl5J8ekAKTFAYIZs/o2yuYjUnwhIdwAGqeprT0xzzirE'
    'fOFvkGlHswxCqBjcI3/XTgpVrohlBPIgNZKAWgIQk0D76vEHa/dizyFSrRbeetqecoksajVpjZOlbhDCU2ZU8iuL3RpCuTlKDH8f'
    '0FDApfkUy+dCZO0rSCDxZyyjfr49i76oGyM6Vg1NS1gt1IdvS70c2cVjL8K+cBF3bo6umSPcfm5/vDt5+Q6IA5Z2EphqPXuaRFNx'
    'MWqTZblQx5qIJE5usG1fWgi2NuMq/FYDQKyaOlk8W0vPso+gc1ScZkefEiyajTlJrpaW+sYCVNvESNQtqxHNEcbSPDoWv6j/f2ZW'
    'Q7MpdL4jEeWrOcpbpN69MeAN3cAdMyDYZmUO2ML7CDyda3jQOqcNPcBvNLO/IDoHPpKLbp0XVAeJ+YNQnhvxyQXnibg4oiIa3syI'
    'Kp55znXBJOJ1vfquw6fMQHR0aabE5y7UvdHwCyX7je7BuS2nemZLaUifnkhFMzZyG6SOKM4oNhJep1FzkXSKYL9nl6nM8jQ6VktV'
    'c6x6FvimiRZLV/dSz20CmeXK20bO+VaC/9JsjAxwU3Tj+8FQB9cNDUsYj37ekvY2AUBsnhkV1PPCwVR2Exy2UATRUNUe0l8ueWvQ'
    'jtW9R1NPS937NX5fEs7iOp7ianW+VWsc7P1C/6/kO1TWXZ300PL4ezlPq4kckYcmKuNij+jgs8pnCC/ozuzjKdGCEF4JHq5lRYql'
    'kYtxh1YVtWS3Y1mNjjSDqC2dHqamevopx4XtSaiWP0xlu22l1IA5GmG+IUvp/f/7VKXbjpL2wYu0ax+tCm1WyvY46GegmDMJDYET'
    'GijYPscMZUYj5Be2POeLG9EdtfWtyMDIvnQAbxUGRppy03ujqA9ZCm8hXcloPRzSA8uGDYWh9EotQJb0AL5FYG5FPbLIBACyMDq1'
    'zl44zx31yxPdUa/lTBc3z2psc1+RpLEnlRoO5RbeUnSku5KR4oQV9BOFk/7USdxUSfCQR5vUTQoWeHEYF20PLx48u3gZNBZfPbeY'
    '2CtWN4V1R8LSEF4fQqh65L7YVeC0kI4B/L4A+P1zAHsFQIS9OKGSxA1RqkV+eTBOx8xWntDbEDz5EZfcbXZ1z314DD7k0319wI4K'
    'aFySWypydYzk64Lk668imY4A0uzUivjkQpyfw5TQh/YHDZH7NBIYVSfx+k5dLdyPFIvwwwlj7/P+Aj6Wp/oStnphRK9XmC6YnwN9'
    '1Di7eeaoethWiSo3C9QiQIlJVem4wxOBjdSnyTQAQmn3TtRmBXVM1i2d4PDQgQ5ezfTj3UV/Z05eOyKdMiGKweib3QHQpBeD8dT5'
    'OrJpK4pPpqY+tbCYZBsCQVXKxXZRFNeZvOE7ZEJ1bNXMRppZioi6A0KDLi2u4dLtv8GoLxivq8X19OmJlDCAGT6hyOaL1+ML9JGN'
    'Sv6vyqewCn2AjTx1OulRmSiivYNEOiguWzfK4NpOkG8aZa8x4b0q1+wd0N7Xtb39wY/HThdDapwtGkw5MKU8dYnLmcrniIq2g+La'
    'sluwcGbnH6YYqEhtQ8SE7R1GkkQS1y9Gv9dEr2WVYGyEDS5+Xu+NC8t3VSY9wluAaC7S55AjA5C28NFrUbThiirdAsNhNa/IuRsv'
    'WRT/ok6zSSVam7eArXmDienslfP6dRg2BG0NOVkDL50WVtHcY96T4dVWz3S5mlEueFs30bsTjRvmtRaNs31P8ZLLc2vKV16Gw0hu'
    'LLv5tHzvpXlsjUpEW+SrWmyHFvgNqQrCrnH2bF5HoihC50iPvd3BaFplVBG3dE54YHqkXvXeyhMiftE3mMF0A2Gt66/ePrTccMNH'
    '4NVLaEDnij69iWbXEZo6mf233yDFzIDpwZ6nsEPT6zN7xoZNVVHK6uCN9aPo1tqRkOmKsWdEbr72yj+sozqxf3RRU9YzLes94nmo'
    'SlHk3JDbdhSepQeFoTInWM4K6fpp/STWVMIrfk7nXHvFYulN9GIc1YVIvJ3qRUExHj3uxOPj4263Oyv5IbzOpxhl6lnnzN4vsQ45'
    '9A6prDTqOCO0Gd9M9B7BTGdEHx+HAp/Vi5CPNcLx3ysoF2d2pzQFLn3aGKE6ZBGSobQ9NZlgxCXlTVnF/xt3fuazPi0AAA=='
  )),
}
ok = True
for path, (sha, parts) in FILES.items():
    data = gzip.decompress(base64.b64decode(''.join(parts)))
    good = hashlib.sha256(data).hexdigest() == sha
    ok &= good
    os.makedirs(os.path.dirname(os.path.join(ROOT, path)), exist_ok=True)
    open(os.path.join(ROOT, path), 'wb').write(data)
    print('%s  %s  %s' % ('ok ' if good else 'BAD', sha, path))
os.chdir(ROOT)
print('\nall files written and checked' if ok else '\nCHECKSUM MISMATCH')
!nproc; lscpu | grep -E 'Model name|Vendor ID'

## 2. Quick verification (~10–15 min)

In [ ]:
!python3 verify/verify_result.py quick --out /content/verify_run/quick

## 3. The whole critical range N = 419…478 (a few hours)
Logs go to Google Drive so they survive a disconnect. After a disconnect, run cell 1 and this cell again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/g3_7_verify_run'
!python3 verify/verify_result.py critical --out {OUT}/critical

## 4. (optional) Every N = 1…478

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/g3_7_verify_run'
!python3 verify/verify_result.py full --out {OUT}/full

## 5. (optional) The independent Rust implementation at N = 473 and 474

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
import os; os.environ['PATH'] += ':/root/.cargo/bin'
!python3 verify/verify_result.py quick --rust --out /content/verify_run/quick

## What the result means
* **quick** confirms the upper bound (the certificate is checked directly from the definition) and that the two
  decisive values N = 473 and 474 behave as claimed, with full count vectors matching the published table.
* **critical** (with Korsky's bound) or **full** (without it) re-establishes the lower bound: no admissible 7-set
  with maximum ≤ 473.
* If you post about the result, say what you ran. The Erdős Problems forum asks that claims be verified by a human
  and that AI assistance be disclosed.